# Limpeza de `onboarding.csv`

**Objetivo:** gerar o questionário de entrada pronto para análise em `dados/processados/onboarding_limpa.parquet`.

**Regras aplicadas** (decisões de 07/11/2026; o diagnóstico está no notebook 17):

| Passo | Regra |
|---|---|
| 1 | `respondido_em` como data (`data_resposta`) |
| 2 | Objetivo e como conheceu sem conteúdo (`nada`, `ok`, `x`, `.`, `-`, `asdf`, `kkkk`, `sla`, `nao sei`, só emoji) e vazios viram ausente; `nivel` vazio vira ausente |
| 3 | `dias_por_semana` vira número de 1 a 7 (`N`, `Nx`, `N vezes` → N; `todos` → 7) |
| 4 | Como conheceu padronizado para as 20 respostas frequentes; `influenciador` com os 6 @ corrigidos |

**Atenção ao usar:** o como conheceu repete o `canal_aquisicao` do cadastro: não é variável nova; aproveite só o
`influenciador`. O objetivo continua em texto livre (com as palavras da pessoa) para a extração com IA generativa.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.

In [1]:
ARQUIVO = "onboarding.csv"
NOME_SAIDA = "onboarding_limpa.parquet"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)
    return t.isin(SEM_CONTEUDO) | so_simbolo

usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

## 0. Leitura

In [2]:
onb = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(onb) == 63_929
assert onb["onboarding_id"].is_unique and onb["user_id"].is_unique
base = registrar("0. Arquivo bruto", "onboarding.csv sem filtro", onb.copy())

0. Arquivo bruto: 63,929 linhas (0 removidas, 0 alteradas)


## 1. Data da resposta

In [3]:
base["data_resposta"] = pd.to_datetime(base["respondido_em"], format="%Y-%m-%d")
assert base["data_resposta"].notna().all() and base["data_resposta"].max() <= CORTE
base = registrar("1. Data da resposta", "respondido_em como data", base, len(base))

1. Data da resposta: 63,929 linhas (0 removidas, 63,929 alteradas)


## 2. Respostas sem conteúdo

In [4]:
alteradas = 0
for col in ["objetivo_texto", "como_conheceu"]:
    lixo = sem_conteudo(base[col])
    vazio = base[col].str.strip() == ""
    print(f"{col}: {lixo.sum():,} sem conteúdo e {vazio.sum():,} vazios viram ausente")
    alteradas += int(lixo.sum())
    base[col] = base[col].str.strip().where(~(lixo | vazio), None)
assert alteradas == 13_768 + 11_515
base["nivel"] = base["nivel"].where(base["nivel"] != "", None)
assert set(base["nivel"].dropna()) == {"iniciante", "intermediario", "avancado"}
base = registrar("2. Respostas sem conteúdo", "objetivo e como conheceu sem conteúdo viram ausente; nivel vazio vira ausente",
                 base, alteradas)

objetivo_texto: 13,768 sem conteúdo e 5,467 vazios viram ausente
como_conheceu: 11,515 sem conteúdo e 4,586 vazios viram ausente
2. Respostas sem conteúdo: 63,929 linhas (0 removidas, 25,283 alteradas)


## 3. Dias por semana

In [5]:
dias = base["dias_por_semana"].str.strip().str.lower()
numero = dias.str.extract(r"^([1-7])(?:\s*x|\s*vezes)?$")[0].where(dias != "todos", "7")
assert (numero.isna() == (dias == "")).all()
fora_padrao = int((dias.ne("") & ~dias.str.fullmatch(r"[1-7]")).sum())
base["dias_por_semana"] = pd.to_numeric(numero).astype("Int64")
display(base["dias_por_semana"].value_counts(dropna=False).sort_index().to_frame("contas"))
assert fora_padrao == 24_034
base = registrar("3. Dias por semana", "N, Nx, N vezes -> N; todos -> 7; vazio -> ausente", base, fora_padrao)

,contas
dias_por_semana,
1,4502
2,12577
3,17161
4,12446
5,6179
6,2204
7,2492
<NA>,6368


3. Dias por semana: 63,929 linhas (0 removidas, 24,034 alteradas)


## 4. Como conheceu e influenciador

Mesma regra do diagnóstico: espaço em volta do @ corrigido; cada grafia vai para a resposta frequente (≥ 100 contas) mais
parecida sem acento (`difflib`, corte 0,8) e cada @ para o influenciador frequente (≥ 500) mais parecido. A conferência é a
do diagnóstico: toda resposta corrigida cai no mesmo canal de aquisição da conta.

In [6]:
import difflib
import unicodedata

def sem_acento(t):
    return "".join(ch for ch in unicodedata.normalize("NFD", t) if unicodedata.category(ch) != "Mn")

cc = base["como_conheceu"].fillna("")
cc = cc.str.replace(r"@\s+(\w)", r"@\1", regex=True).str.replace(r"(\w)@(\w)", r"\1 @\2", regex=True)
handle = cc.str.extract(r"@(\w+)")[0]
contagem_handle = handle.value_counts()
HANDLES = list(contagem_handle[contagem_handle >= 500].index)
mapa_handle = {h: (difflib.get_close_matches(h, HANDLES, n=1, cutoff=0.8) or [None])[0] for h in handle.dropna().unique()}
assert all(v is not None for v in mapa_handle.values())
base["influenciador"] = handle.map(mapa_handle)

forma = cc.str.lower().str.replace(r"@\w+", "@X", regex=True)
contagem = forma[forma != ""].value_counts()
POR_CHAVE = {}
for t, n in contagem[contagem >= 100].items():
    POR_CHAVE.setdefault(sem_acento(t), t)
CHAVES = list(POR_CHAVE)
def canonica(t):
    k = sem_acento(t)
    if k in POR_CHAVE:
        return POR_CHAVE[k]
    perto = difflib.get_close_matches(k, CHAVES, n=1, cutoff=0.8)
    return POR_CHAVE[perto[0]] if perto else None
mapa = {t: canonica(t) for t in contagem.index}
assert all(v is not None for v in mapa.values())
padrao = forma.map(mapa)
corrigidas = int((padrao.notna() & (padrao != forma)).sum())

canal_aq = base["user_id"].map(usuarios["canal_aquisicao"])
canal_resposta = canal_aq[padrao.notna()].groupby(padrao.dropna()).agg(lambda s: s.value_counts().index[0])
assert (padrao.dropna().map(canal_resposta) == canal_aq[padrao.notna()]).all()   # cada resposta num único canal
assert corrigidas == 4_088

# devolve o nome do influenciador no lugar do @X
base["como_conheceu"] = [p.replace("@X", "@" + h) if isinstance(p, str) and isinstance(h, str) else p
                         for p, h in zip(padrao, base["influenciador"])]
print(f"{len(contagem)} grafias -> {len(set(mapa.values()))} respostas; corrigidas: {corrigidas:,}; "
      f"com influenciador: {base['influenciador'].notna().sum():,} -> {base['influenciador'].value_counts().to_dict()}")
base = registrar("4. Como conheceu", "20 respostas frequentes; influenciador com os 6 @ corrigidos", base, corrigidas)

300 grafias -> 20 respostas; corrigidas: 4,088; com influenciador: 7,793 -> {'julianamove': 1345, 'paulocorre': 1328, 'runwithbia': 1326, 'marifitness': 1279, 'tiagopace': 1271, 'carolfit': 1244}
4. Como conheceu: 63,929 linhas (0 removidas, 4,088 alteradas)


## 5. Conferências

In [7]:
conferencias = {
    "linhas": (len(base), 63_929),
    "user_id repetido": (int(base["user_id"].duplicated().sum()), 0),
    "contas fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios.index)).sum()), 0),
    "contas de usuarios_limpa sem questionário": (len(set(usuarios.index) - set(base["user_id"])), 0),
    "respostas de como conheceu": (int(base["como_conheceu"].str.replace(r"@\w+", "@X", regex=True).nunique()), 20),
    "influenciadores": (int(base["influenciador"].nunique()), 6),
    "dias por semana fora de 1 a 7": (int((~base["dias_por_semana"].dropna().between(1, 7)).sum()), 0),
}

In [8]:
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
linhas,63929,63929,True
user_id repetido,0,0,True
contas fora de usuarios_limpa,0,0,True
contas de usuarios_limpa sem questionário,0,0,True
respostas de como conheceu,20,20,True
influenciadores,6,6,True
dias por semana fora de 1 a 7,0,0,True


## 6. Saídas e integridade dos brutos

In [9]:
COLUNAS = ["onboarding_id", "user_id", "data_resposta", "objetivo_texto", "como_conheceu", "influenciador", "nivel",
           "dias_por_semana"]
limpa = base[COLUNAS].reset_index(drop=True)

for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / NOME_SAIDA
limpa.to_parquet(destino, index=False)
relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)
pd.DataFrame(registro).to_csv(SAIDAS / f"limpeza_{Path(ARQUIVO).stem}_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\onboarding_limpa.parquet: 63,929 linhas x 8 colunas


,tipo,ausentes
onboarding_id,object,0
user_id,object,0
data_resposta,datetime64[ns],0
objetivo_texto,object,19235
como_conheceu,object,16101
influenciador,object,56136
nivel,object,4513
dias_por_semana,Int64,6368


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,onboarding.csv sem filtro,63929,0,0
1,1. Data da resposta,respondido_em como data,63929,0,63929
2,2. Respostas sem conteúdo,objetivo e como conheceu sem conteúdo viram ausente; nivel vazio v...,63929,0,25283
3,3. Dias por semana,"N, Nx, N vezes -> N; todos -> 7; vazio -> ausente",63929,0,24034
4,4. Como conheceu,20 respostas frequentes; influenciador com os 6 @ corrigidos,63929,0,4088


dados/brutos/ inalterado: True
